# 04. 파이썬 기초 - pandas 데이터분석

`pandas` 는 표(엑셀 같은) 데이터를 다루는 핵심 라이브러리입니다.
이 프로젝트의 넷플릭스·K-Pop 분석이 모두 pandas 로 작성되어 있습니다.

**다루는 내용**
1. Series 와 DataFrame
2. '딕셔너리들의 리스트' → DataFrame
3. 열 선택 / 행 필터링 (loc)
4. 정렬, value_counts
5. 결측치 처리
6. groupby 집계
7. 날짜 변환 (to_datetime)

In [1]:
import pandas as pd   # 관례적으로 pd 라는 별칭 사용
print('pandas 버전:', pd.__version__)

pandas 버전: 2.2.2


## 1. Series 와 DataFrame

- **Series**: 1차원 (한 개의 열)
- **DataFrame**: 2차원 표 (여러 열)

In [2]:
# Series: 리스트에 인덱스가 붙은 형태
prices = pd.Series([25000, 30000, 28000])
print(prices)
print('평균:', prices.mean())

0    25000
1    30000
2    28000
dtype: int64
평균: 27666.666666666668


## 2. '딕셔너리들의 리스트' → DataFrame 

01편에서 배운 구조가 그대로 표로 변환됩니다.
스크래핑 결과 `pd.DataFrame(books)` 가 바로 이것입니다.

In [3]:
books = [
    {'title': '파이썬 입문', 'author': '홍길동', 'price': 25000, 'publisher': 'A출판'},
    {'title': '데이터 분석', 'author': '김철수', 'price': 30000, 'publisher': 'B출판'},
    {'title': '웹 스크래핑', 'author': '이영희', 'price': 28000, 'publisher': 'A출판'},
    {'title': '머신러닝',   'author': '박민수', 'price': 33000, 'publisher': 'C출판'},
]

df = pd.DataFrame(books)
df

,title,author,price,publisher
0,파이썬 입문,홍길동,25000,A출판
1,데이터 분석,김철수,30000,B출판
2,웹 스크래핑,이영희,28000,A출판
3,머신러닝,박민수,33000,C출판


In [5]:
# 데이터 훑어보기 (분석 시작 시 항상 확인하는 것들)
print('행 x 열:', df.shape)      # (4, 4)
print('\n컬럼:', list(df.columns))
print('\n--- 앞부분 미리보기 ---')
print(df.head(2))                # 위에서 2줄
print('\n--- 요약 통계 ---')
df['price'].describe()    # 숫자열 통계

행 x 열: (4, 4)

컬럼: ['title', 'author', 'price', 'publisher']

--- 앞부분 미리보기 ---
    title author  price publisher
0  파이썬 입문    홍길동  25000       A출판
1  데이터 분석    김철수  30000       B출판

--- 요약 통계 ---


count        4.000000
mean     29000.000000
std       3366.501646
min      25000.000000
25%      27250.000000
50%      29000.000000
75%      30750.000000
max      33000.000000
Name: price, dtype: float64

## 3. 열 선택 / 행 필터링

- 열 선택: `df['컬럼']` 또는 `df[['컬럼1','컬럼2']]`
- 행 필터: `df[조건]` — 조건이 True인 행만 남김

In [7]:
# 한 개 열 선택 → Series
print(df['title'])

print('-' * 30)

# 여러 열 선택 → DataFrame (대괄호 두 겹)
df[['title', 'price']]

0    파이썬 입문
1    데이터 분석
2    웹 스크래핑
3      머신러닝
Name: title, dtype: object
------------------------------


,title,price
0,파이썬 입문,25000
1,데이터 분석,30000
2,웹 스크래핑,28000
3,머신러닝,33000


In [8]:
# 조건 필터링: 가격이 28000 이상인 책
condition = df['price'] >= 28000
print(condition)               # True/False 의 Series
print('-' * 30)
df[condition]           # True인 행만


0    False
1     True
2     True
3     True
Name: price, dtype: bool
------------------------------


,title,author,price,publisher
1,데이터 분석,김철수,30000,B출판
2,웹 스크래핑,이영희,28000,A출판
3,머신러닝,박민수,33000,C출판


In [9]:

# loc: [행조건, 열목록] 을 한 번에 (실제 분석 코드에서 자주 사용)
print('-' * 30)
df.loc[df['price'] >= 28000, ['title', 'price']]

------------------------------


,title,price
1,데이터 분석,30000
2,웹 스크래핑,28000
3,머신러닝,33000


## 4. 정렬과 개수 세기

In [10]:
# 가격 높은 순 정렬
print(df.sort_values('price', ascending=False))

print('-' * 30)

# value_counts(): 값별 개수 (출판사별 도서 수) — 분석에서 매우 자주 사용
print(df['publisher'].value_counts())

    title author  price publisher
3    머신러닝    박민수  33000       C출판
1  데이터 분석    김철수  30000       B출판
2  웹 스크래핑    이영희  28000       A출판
0  파이썬 입문    홍길동  25000       A출판
------------------------------
publisher
A출판    2
B출판    1
C출판    1
Name: count, dtype: int64


## 5. 결측치(빠진 값) 처리

실제 데이터에는 빈 값(NaN)이 흔합니다. 넷플릭스·K-Pop 데이터도 결측이 많았습니다.

In [12]:
import numpy as np

# 일부러 결측치가 있는 데이터 생성
df2 = pd.DataFrame({
    'name': ['A', 'B', 'C', 'D'],
    'height': [172, np.nan, 168, 180],   # B의 키가 결측
})
df2

,name,height
0,A,172.0
1,B,NaN
2,C,168.0
3,D,180.0


In [13]:

print('결측치 개수:\n', df2.isna().sum())     # 열별 결측 개수
print('\n결측 행 제거:\n', df2.dropna())        # NaN 있는 행 삭제
print('\n결측을 평균으로 채움:\n', df2.fillna(df2['height'].mean()))

결측치 개수:
 name      0
height    1
dtype: int64

결측 행 제거:
   name  height
0    A   172.0
2    C   168.0
3    D   180.0

결측을 평균으로 채움:
   name      height
0    A  172.000000
1    B  173.333333
2    C  168.000000
3    D  180.000000


## 6. groupby — 그룹별 집계 ⭐

'출판사별 평균 가격' 처럼 **그룹으로 묶어 계산** 합니다.
K-Pop 분석의 '소속사별 평균 데뷔 나이' 가 이 문법입니다.

In [15]:
# 출판사별 평균 가격
print(df.groupby('publisher')['price'].mean())
df

publisher
A출판    26500.0
B출판    30000.0
C출판    33000.0
Name: price, dtype: float64


,title,author,price,publisher
0,파이썬 입문,홍길동,25000,A출판
1,데이터 분석,김철수,30000,B출판
2,웹 스크래핑,이영희,28000,A출판
3,머신러닝,박민수,33000,C출판


In [16]:

print('-' * 30)

# 출판사별 도서 수 + 평균 가격 (여러 집계 한 번에)
summary = df.groupby('publisher')['price'].agg(['count', 'mean'])
summary

------------------------------


,count,mean
publisher,,
A출판,2,26500.0
B출판,1,30000.0
C출판,1,33000.0


## 7. 날짜 다루기 — to_datetime

문자열로 된 날짜를 날짜형으로 바꾸면 연/월 추출, 기간 계산이 쉬워집니다.
K-Pop 분석에서 데뷔 나이를 계산할 때 사용했습니다.

In [20]:
df3 = pd.DataFrame({'debut': ['26/08/2014', '31/10/2015', '11/10/2017']})
print(df3['debut'].dtype)
df3

object


,debut
0,26/08/2014
1,31/10/2015
2,11/10/2017


In [21]:

# 문자열 → 날짜형 (dd/mm/yyyy 형식이므로 dayfirst=True)
df3['debut'] = pd.to_datetime(df3['debut'], dayfirst=True)
print(df3['debut'].dtype)

datetime64[ns]


In [23]:

# 날짜형이 되면 .dt 로 연/월/일 추출 가능
df3['year'] = df3['debut'].dt.year
print(df3['year'].dtype)
df3


int32


,debut,year
0,2014-08-26,2014
1,2015-10-31,2015
2,2017-10-11,2017


In [ ]:

# 두 날짜의 차이 (일수) 계산
gap = pd.to_datetime('2020-01-01') - pd.to_datetime('2014-08-26')
print('\n기간(일):', gap.days)


기간(일): 1954


## 정리
- **DataFrame**: '딕셔너리들의 리스트' → 표
- **훑어보기**: `shape`, `head()`, `columns`, `describe()`
- **선택/필터**: `df['col']`, `df.loc[조건, 열목록]`
- **정렬/집계**: `sort_values`, `value_counts`, `groupby().agg()`
- **결측치**: `isna()`, `dropna()`, `fillna()`
- **날짜**: `to_datetime()`, `.dt.year`

다음: `05python_basic_웹요청과파싱.ipynb` (requests·BeautifulSoup)